# AD688 Group Project — Comprehensive EDA (v2)
**Course:** MET AD688 · Boston University  
**Dataset:** Lightcast 2024 — Data / Analytics / Tech Job Postings  

### Dataset Context
The Lightcast 2024 dataset is **pre-filtered to data, analytics, and tech occupations** (~72k postings). `TITLE` is a hash ID — the readable job title is in `TITLE_RAW`. Salary columns are ~55-58% missing by design (not all employers post salary).

### Research Tracks
| Track | Research Question |
|---|---|
| **A** | AI/ML vs. Traditional Analyst/BI — salary premium and growth trends |
| **B** | Red/Blue State Influence — do state politics shape tech salaries/hiring? |
| **C** | Salary, Geography & Remote Work — what factors predict higher pay? |

---
## 0. Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["gdown", "plotly", "missingno", "kaleido"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages ready.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import missingno as msno
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")
print("Libraries loaded.")

---
## 1. Load Dataset

In [ ]:
import gdown, os
FILE_ID = "1V2GCHGt2dkFGqVBeoUFckU4IhUgk4ocQ"
OUTPUT  = "lightcast_2024.csv"
if not os.path.exists(OUTPUT):
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", OUTPUT, quiet=False)
else:
    print(f"File already exists: {OUTPUT}")
df_raw = pd.read_csv(OUTPUT, low_memory=False)
print(f"Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head(3)

---
## 2. Data Overview

In [ ]:
print(f"Shape: {df_raw.shape}")
print("\nDtype summary:")
print(df_raw.dtypes.value_counts())
print("\nAll columns:")
for c in df_raw.columns:
    print(f"  {c}  [{df_raw[c].dtype}]")

In [ ]:
# Missing value overview
# Note: SALARY/SALARY_FROM/SALARY_TO are ~55-58% missing — normal for job postings.
# LIGHTCAST_SECTORS ~75%, MAX_EDULEVELS ~77% missing — use alternatives.
# We protect salary columns from the 50% drop threshold in cleaning.
missing = pd.DataFrame({
    "missing_count": df_raw.isnull().sum(),
    "missing_pct":   df_raw.isnull().mean() * 100
}).sort_values("missing_pct", ascending=False)
print(missing[missing["missing_count"] > 0].to_string())

In [ ]:
plt.figure(figsize=(14, 6))
msno.heatmap(df_raw, cmap="coolwarm")
plt.title("Missing Values Correlation Heatmap", fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Data Cleaning

In [ ]:
df = df_raw.copy()

# 3.1  Drop tracking / redundant columns
# Keep: NAICS_2022_6_NAME, SOC_2021_4_NAME, and all _NAME variants
# Drop: hash IDs, URLs, older hierarchy codes (NAICS2-6 shorthand, SOC_2/3/5)
columns_to_drop = [
    "ID", "URL", "ACTIVE_URLS", "ACTIVE_SOURCES_INFO",
    "DUPLICATES", "LAST_UPDATED_TIMESTAMP", "LAST_UPDATED_DATE",
    "NAICS2", "NAICS3", "NAICS4", "NAICS5", "NAICS6",
    "SOC_2", "SOC_2_NAME", "SOC_3", "SOC_3_NAME", "SOC_5", "SOC_5_NAME",
]
existing_drops = [c for c in columns_to_drop if c in df.columns]
df.drop(columns=existing_drops, inplace=True)
print(f"3.1 Dropped {len(existing_drops)} tracking columns. Remaining: {df.shape[1]}")

In [ ]:
# 3.2  Drop high-missing columns — PROTECT salary & education columns
# FIX: Without protection, the 50% threshold drops SALARY, SALARY_FROM, SALARY_TO
# (all >55% missing), leaving COL_SALARY = None and breaking all salary analysis.
PROTECT_COLS = [
    "SALARY", "SALARY_FROM", "SALARY_TO", "ORIGINAL_PAY_PERIOD",
    "MIN_YEARS_EXPERIENCE", "MAX_YEARS_EXPERIENCE",
    "LIGHTCAST_SECTORS", "LIGHTCAST_SECTORS_NAME",
    "MIN_EDULEVELS", "MIN_EDULEVELS_NAME",
    "MAX_EDULEVELS", "MAX_EDULEVELS_NAME",
]
protect_existing = [c for c in PROTECT_COLS if c in df.columns]
non_protected    = [c for c in df.columns if c not in protect_existing]

keep_non_prot = [c for c in non_protected if df[c].isnull().mean() < 0.5]
cols_to_keep  = keep_non_prot + protect_existing
dropped_count = df.shape[1] - len(cols_to_keep)
df = df[cols_to_keep]
print(f"3.2 Dropped {dropped_count} high-missing columns (salary/edu protected). Remaining: {df.shape[1]}")

In [ ]:
# 3.3  Remove duplicate postings (by title + company + state + date)
dup_subset = [c for c in ["TITLE_RAW", "COMPANY_NAME", "STATE_NAME", "POSTED"] if c in df.columns]
before = len(df)
df.drop_duplicates(subset=dup_subset, keep="first", inplace=True)
print(f"3.3 Removed {before - len(df):,} duplicates. Remaining rows: {len(df):,}")

In [ ]:
# 3.4  Build unified salary column (SALARY_MID)
# Strategy: midpoint of SALARY_FROM/TO, fall back to SALARY.
# Do NOT impute missing with median — that distorts distributions.
# All salary plots use .dropna() / filtered subsets.
if "SALARY_FROM" in df.columns and "SALARY_TO" in df.columns:
    df["SALARY_MID"] = df[["SALARY_FROM", "SALARY_TO"]].mean(axis=1)
    if "SALARY" in df.columns:
        df["SALARY_MID"] = df["SALARY_MID"].fillna(df["SALARY"])
    print("Created SALARY_MID from (SALARY_FROM + SALARY_TO) / 2")
elif "SALARY" in df.columns:
    df["SALARY_MID"] = df["SALARY"]
    print("Using SALARY as SALARY_MID")
else:
    print("WARNING: No salary columns found. Check columns_to_drop in step 3.1.")

if "SALARY_MID" in df.columns:
    n_sal = df["SALARY_MID"].notna().sum()
    sal_c = df[(df["SALARY_MID"] > 10_000) & (df["SALARY_MID"] < 500_000)]["SALARY_MID"]
    print(f"Postings with salary: {n_sal:,} ({n_sal/len(df)*100:.1f}%)")
    print(f"Median salary: ${sal_c.median():,.0f}  |  Mean: ${sal_c.mean():,.0f}")

In [ ]:
# 3.5  Fill categorical missing values
fill_unknown = [
    "NAICS_2022_6_NAME", "NAICS_2022_2_NAME", "SOC_2021_4_NAME",
    "REMOTE_TYPE_NAME", "STATE_NAME", "MIN_EDULEVELS_NAME",
    "LOT_V6_OCCUPATION_NAME", "LOT_OCCUPATION_NAME",
    "EMPLOYMENT_TYPE_NAME", "LIGHTCAST_SECTORS_NAME",
]
for col in fill_unknown:
    if col in df.columns:
        df[col].fillna("Unknown", inplace=True)

print(f"Final cleaned shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

In [ ]:
# 3.6  Column mapping
# IMPORTANT: TITLE = hash ID (not useful for display)
#            TITLE_RAW = actual job title text (e.g. 'Data Analyst')

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    for cand in candidates:
        for col in df.columns:
            if cand.upper() in col.upper():
                return col
    return None

COL_TITLE      = find_col(df, ["TITLE_RAW", "TITLE_NAME"])  # NOT "TITLE" (hash)
COL_OCCUPATION = find_col(df, ["LOT_V6_OCCUPATION_NAME", "LOT_OCCUPATION_NAME", "SOC_2021_4_NAME"])
COL_INDUSTRY   = find_col(df, ["NAICS_2022_6_NAME", "NAICS_2022_4_NAME", "NAICS_2022_2_NAME"])
COL_IND_BROAD  = find_col(df, ["NAICS_2022_2_NAME"])
COL_STATE      = find_col(df, ["STATE_NAME", "STATE"])
COL_REMOTE     = find_col(df, ["REMOTE_TYPE_NAME", "REMOTE_TYPE"])
COL_SALARY     = find_col(df, ["SALARY_MID", "SALARY", "SALARY_FROM"])
COL_COMPANY    = find_col(df, ["COMPANY_NAME", "COMPANY_RAW"])  # NOT "COMPANY" (numeric ID)
COL_SKILLS     = find_col(df, ["SKILLS_NAME", "SKILL_NAME", "SKILLS"])
COL_SOC        = find_col(df, ["SOC_2021_4_NAME", "SOC_2021_5_NAME"])
COL_POSTED     = find_col(df, ["POSTED", "POSTING_DATE"])
COL_EDU        = find_col(df, ["MIN_EDULEVELS_NAME", "EDUCATION_LEVELS_NAME"])  # NOT MAX (77% missing)
COL_EXP        = find_col(df, ["MIN_YEARS_EXPERIENCE", "MAX_YEARS_EXPERIENCE"])
COL_EMPTYPE    = find_col(df, ["EMPLOYMENT_TYPE_NAME"])

print("Column mapping (v2 — corrected):")
for name, val in [
    ("Job Title (TITLE_RAW)", COL_TITLE), ("Occupation category", COL_OCCUPATION),
    ("Industry (6-digit)",   COL_INDUSTRY), ("Industry (broad 2-digit)", COL_IND_BROAD),
    ("State",                COL_STATE),   ("Remote type",   COL_REMOTE),
    ("Salary (SALARY_MID)",  COL_SALARY),  ("Company name",  COL_COMPANY),
    ("Skills",               COL_SKILLS),  ("SOC occupation",COL_SOC),
    ("Posted date",          COL_POSTED),  ("Min education", COL_EDU),
    ("Experience (yrs)",     COL_EXP),     ("Employment type", COL_EMPTYPE),
]:
    status = "OK" if val else "NOT FOUND"
    print(f"  {name:28s} -> {str(val):35s} [{status}]")

---
## 4. Basic EDA — Market Overview

In [ ]:
# 4.1  Key categorical distributions
key_cats = [c for c in [COL_OCCUPATION, COL_IND_BROAD, COL_STATE, COL_REMOTE, COL_EDU, COL_EMPTYPE] if c]
for c in key_cats:
    vc = df[c].value_counts()
    print(f"\n{c} (unique={vc.shape[0]}):")
    print(vc.head(10).to_string())

In [ ]:
# 4.2  Occupation category distribution
if COL_OCCUPATION:
    occ_counts = df[COL_OCCUPATION].value_counts().reset_index()
    occ_counts.columns = ["Occupation", "Postings"]
    fig = px.bar(occ_counts, x="Postings", y="Occupation", orientation="h",
                 title="Job Postings by Occupation Category (Lightcast 2024)",
                 color="Postings", color_continuous_scale="Blues")
    fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=450)
    fig.show()

In [ ]:
# 4.3  Top job titles (TITLE_RAW)
if COL_TITLE:
    top_titles = df[COL_TITLE].value_counts().head(25).reset_index()
    top_titles.columns = ["Job Title", "Postings"]
    fig = px.bar(top_titles, x="Postings", y="Job Title", orientation="h",
                 title="Top 25 Job Titles by Posting Volume (TITLE_RAW)",
                 color="Postings", color_continuous_scale="Teal")
    fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=700)
    fig.show()

In [ ]:
# 4.4  Top industries (6-digit NAICS, excluding Unclassified)
if COL_INDUSTRY:
    top_ind = (df[df[COL_INDUSTRY] != "Unclassified Industry"][COL_INDUSTRY]
               .value_counts().head(20).reset_index())
    top_ind.columns = ["Industry", "Postings"]
    fig = px.bar(top_ind, x="Postings", y="Industry", orientation="h",
                 title="Top 20 Industries Hiring Data/Tech Workers (2024)",
                 color="Postings", color_continuous_scale="Blues")
    fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=650)
    fig.show()

In [ ]:
# 4.5  Salary distribution (non-null subset only)
if COL_SALARY:
    sal = df[COL_SALARY].dropna()
    sal = sal[(sal > 10_000) & (sal < 500_000)]
    pct = len(sal) / len(df) * 100
    print(f"Postings with valid salary: {len(sal):,} ({pct:.1f}% of total)")
    print(f"Median: ${sal.median():,.0f} | Mean: ${sal.mean():,.0f} | Std: ${sal.std():,.0f}")
    print(f"10th pct: ${sal.quantile(0.1):,.0f} | 90th pct: ${sal.quantile(0.9):,.0f}")

    fig = px.histogram(sal, nbins=80,
                       title=f"Annual Salary Distribution ({len(sal):,} postings with salary = {pct:.0f}%)",
                       labels={"value": "Annual Salary ($)"},
                       color_discrete_sequence=["steelblue"])
    fig.add_vline(x=sal.median(), line_dash="dash", line_color="red",
                  annotation_text=f"Median ${sal.median():,.0f}")
    fig.add_vline(x=sal.mean(), line_dash="dot", line_color="orange",
                  annotation_text=f"Mean ${sal.mean():,.0f}")
    fig.show()
else:
    print("SALARY_MID not found. Re-check cleaning step 3.2 (salary protection).")

In [ ]:
# 4.6  Salary by occupation category
if COL_SALARY and COL_OCCUPATION:
    df_s = df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000)]
    fig = px.box(df_s, x=COL_OCCUPATION, y=COL_SALARY, color=COL_OCCUPATION,
                 title="Salary by Occupation Category",
                 labels={COL_OCCUPATION: "Occupation", COL_SALARY: "Annual Salary ($)"})
    fig.update_layout(xaxis_tickangle=-20, showlegend=False, height=500)
    fig.show()
    print(df_s.groupby(COL_OCCUPATION)[COL_SALARY]
          .agg(Median="median", Mean="mean", Count="count")
          .sort_values("Median", ascending=False))

In [ ]:
# 4.7  Remote vs On-Site
if COL_REMOTE:
    rc = df[COL_REMOTE].value_counts().reset_index()
    rc.columns = ["Work Type", "Count"]
    fig = px.pie(rc, names="Work Type", values="Count",
                 title="Remote vs. Hybrid vs. On-Site Distribution",
                 color_discrete_sequence=px.colors.qualitative.Set2, hole=0.35)
    fig.show()

    if COL_SALARY:
        df_r = df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000)]
        fig2 = px.box(df_r, x=COL_REMOTE, y=COL_SALARY, color=COL_REMOTE,
                      title="Salary by Work Arrangement",
                      labels={COL_REMOTE: "Work Type", COL_SALARY: "Annual Salary ($)"})
        fig2.update_layout(showlegend=False)
        fig2.show()
        print(df_r.groupby(COL_REMOTE)[COL_SALARY]
              .agg(Median="median", Mean="mean", Count="count")
              .sort_values("Median", ascending=False))

In [ ]:
# 4.8  Geographic distribution
STATE_ABBREV = {
    "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA",
    "Colorado":"CO","Connecticut":"CT","Delaware":"DE","Florida":"FL","Georgia":"GA",
    "Hawaii":"HI","Idaho":"ID","Illinois":"IL","Indiana":"IN","Iowa":"IA",
    "Kansas":"KS","Kentucky":"KY","Louisiana":"LA","Maine":"ME","Maryland":"MD",
    "Massachusetts":"MA","Michigan":"MI","Minnesota":"MN","Mississippi":"MS",
    "Missouri":"MO","Montana":"MT","Nebraska":"NE","Nevada":"NV",
    "New Hampshire":"NH","New Jersey":"NJ","New Mexico":"NM","New York":"NY",
    "North Carolina":"NC","North Dakota":"ND","Ohio":"OH","Oklahoma":"OK",
    "Oregon":"OR","Pennsylvania":"PA","Rhode Island":"RI","South Carolina":"SC",
    "South Dakota":"SD","Tennessee":"TN","Texas":"TX","Utah":"UT",
    "Vermont":"VT","Virginia":"VA","Washington":"WA","West Virginia":"WV",
    "Wisconsin":"WI","Wyoming":"WY","District of Columbia":"DC"
}

if COL_STATE:
    sample_val = str(df[COL_STATE].dropna().iloc[0])
    if len(sample_val) > 2:
        df["STATE_ABBR"] = df[COL_STATE].map(STATE_ABBREV)
    else:
        df["STATE_ABBR"] = df[COL_STATE]

    state_counts = df[COL_STATE].value_counts().reset_index()
    state_counts.columns = ["State", "Postings"]
    state_counts["Abbr"] = state_counts["State"].map(STATE_ABBREV).fillna(state_counts["State"])

    fig = px.choropleth(state_counts.dropna(subset=["Abbr"]),
                        locations="Abbr", locationmode="USA-states",
                        color="Postings", scope="usa",
                        title="Data/Tech Job Postings by State (2024)",
                        color_continuous_scale="Blues")
    fig.show()

    fig2 = px.bar(state_counts.head(15), x="State", y="Postings",
                  title="Top 15 States — Data/Tech Job Postings",
                  color="Postings", color_continuous_scale="Blues")
    fig2.update_xaxes(tickangle=-30)
    fig2.show()

In [ ]:
# 4.9  Posting volume over time
if COL_POSTED:
    df["POSTED_DT"]    = pd.to_datetime(df[COL_POSTED], errors="coerce")
    df["POSTED_MONTH"] = df["POSTED_DT"].dt.to_period("M").astype(str)
    monthly = df.groupby("POSTED_MONTH").size().reset_index(name="Postings")
    monthly = monthly[monthly["POSTED_MONTH"] != "NaT"]
    fig = px.line(monthly, x="POSTED_MONTH", y="Postings",
                  title="Monthly Job Posting Volume — Data/Tech Roles (2024)", markers=True)
    fig.update_xaxes(tickangle=-45)
    fig.show()

In [ ]:
# 4.10  Education level required
if COL_EDU:
    edu_counts = df[COL_EDU].value_counts().reset_index()
    edu_counts.columns = ["Education", "Postings"]
    fig = px.bar(edu_counts, x="Postings", y="Education", orientation="h",
                 title="Minimum Education Required for Data/Tech Jobs",
                 color="Postings", color_continuous_scale="Greens")
    fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=400)
    fig.show()

    if COL_SALARY:
        df_e = df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000)]
        edu_sal = (df_e.groupby(COL_EDU)[COL_SALARY]
                   .agg(Median="median", Count="count")
                   .reset_index().query("Count >= 20")
                   .sort_values("Median"))
        fig2 = px.bar(edu_sal, x="Median", y=COL_EDU, orientation="h",
                      title="Median Salary by Minimum Education Level",
                      labels={"Median": "Median Salary ($)"},
                      color="Median", color_continuous_scale="Greens")
        fig2.show()

In [ ]:
# 4.11  Top hiring companies
if COL_COMPANY:
    top_cos = (df[~df[COL_COMPANY].isin(["Unclassified", "Unknown", np.nan])]
               [COL_COMPANY].value_counts().head(20).reset_index())
    top_cos.columns = ["Company", "Postings"]
    fig = px.bar(top_cos, x="Postings", y="Company", orientation="h",
                 title="Top 20 Hiring Companies for Data/Tech Roles",
                 color="Postings", color_continuous_scale="Oranges")
    fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
    fig.show()

---
## 5. Track A — AI/ML vs. Traditional Analyst/BI Careers
**Research Question:** Do ML/AI roles command higher salaries and grow faster than traditional data analyst/BI roles?

In [ ]:
# 5.1  AI vs Non-AI classification using TITLE_RAW + skills
AI_KEYWORDS = [
    "machine learning", "artificial intelligence", "deep learning",
    "data science", "data scientist", "mlops", "nlp", "natural language",
    "computer vision", "neural network", "llm", "generative ai",
    "ai engineer", "ml engineer", "research scientist", "tensorflow",
    "pytorch", "predictive model", "algorithm"
]

def label_ai(row):
    text = " ".join([str(row.get(c, "")) for c in
                     [COL_TITLE, COL_SOC, COL_OCCUPATION, COL_SKILLS]
                     if c]).lower()
    return "AI / ML" if any(kw in text for kw in AI_KEYWORDS) else "Traditional Analyst / BI"

df["AI_LABEL"] = df.apply(label_ai, axis=1)
print(df["AI_LABEL"].value_counts())
print(f"AI/ML share: {(df['AI_LABEL']=='AI / ML').mean()*100:.1f}%")

In [ ]:
# 5.2  AI vs Non-AI volume
ai_counts = df["AI_LABEL"].value_counts().reset_index()
ai_counts.columns = ["Category", "Postings"]
fig = px.pie(ai_counts, names="Category", values="Postings",
             title="AI/ML vs. Traditional Analyst/BI Job Postings (2024)",
             color_discrete_map={"AI / ML": "#1565C0", "Traditional Analyst / BI": "#78909C"},
             hole=0.4)
fig.show()

In [ ]:
# 5.3  Salary comparison
if COL_SALARY:
    df_ai = df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000)]
    fig = px.box(df_ai, x="AI_LABEL", y=COL_SALARY, color="AI_LABEL",
                 title="Salary: AI/ML vs. Traditional Analyst/BI",
                 labels={"AI_LABEL": "Role Type", COL_SALARY: "Annual Salary ($)"},
                 color_discrete_map={"AI / ML": "#1565C0", "Traditional Analyst / BI": "#78909C"})
    fig.update_layout(showlegend=False)
    fig.show()
    stats = df_ai.groupby("AI_LABEL")[COL_SALARY].agg(Median="median", Mean="mean", Count="count")
    print(stats)
    if "AI / ML" in stats.index and "Traditional Analyst / BI" in stats.index:
        prem = stats.loc["AI / ML","Median"] - stats.loc["Traditional Analyst / BI","Median"]
        print(f"\nAI/ML salary premium: ${prem:,.0f} ({prem/stats.loc['Traditional Analyst / BI','Median']*100:+.1f}%)")

In [ ]:
# 5.4  Posting trends over time
if "POSTED_MONTH" in df.columns:
    ai_monthly = df.groupby(["POSTED_MONTH", "AI_LABEL"]).size().reset_index(name="Postings")
    ai_monthly = ai_monthly[ai_monthly["POSTED_MONTH"] != "NaT"]
    fig = px.line(ai_monthly, x="POSTED_MONTH", y="Postings", color="AI_LABEL",
                  title="AI/ML vs. Traditional Analyst/BI Trends (2024)", markers=True,
                  color_discrete_map={"AI / ML": "#1565C0", "Traditional Analyst / BI": "#78909C"})
    fig.update_xaxes(tickangle=-45)
    fig.show()

In [ ]:
# 5.5  Top job titles for each category
if COL_TITLE:
    for label, color in [("AI / ML", "Blues"), ("Traditional Analyst / BI", "Greys")]:
        top = df[df["AI_LABEL"] == label][COL_TITLE].value_counts().head(20).reset_index()
        top.columns = ["Job Title", "Count"]
        fig = px.bar(top, x="Count", y="Job Title", orientation="h",
                     title=f"Top 20 Job Titles — {label}",
                     color="Count", color_continuous_scale=color)
        fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
        fig.show()

In [ ]:
# 5.6  Remote work arrangement: AI vs Non-AI
if COL_REMOTE:
    remote_ai = df.groupby(["AI_LABEL", COL_REMOTE]).size().reset_index(name="Count")
    remote_ai["Share"] = remote_ai.groupby("AI_LABEL")["Count"].transform(lambda x: x/x.sum()*100)
    fig = px.bar(remote_ai, x="AI_LABEL", y="Share", color=COL_REMOTE, barmode="stack",
                 title="Work Arrangement: AI/ML vs. Traditional Analyst/BI",
                 labels={"Share": "Share (%)", "AI_LABEL": "Role Type"})
    fig.show()

In [ ]:
# 5.7  Top skills by category
if COL_SKILLS:
    from collections import Counter
    def top_skills(series, n=20):
        all_s = []
        for v in series.dropna():
            s = str(v)
            sep = next((c for c in [";",",","|"] if c in s), None)
            all_s.extend([x.strip() for x in s.split(sep)] if sep else [s.strip()])
        return pd.DataFrame(Counter(all_s).most_common(n), columns=["Skill","Count"])

    for label, color in [("AI / ML","Blues"),("Traditional Analyst / BI","Greys")]:
        sk = top_skills(df[df["AI_LABEL"]==label][COL_SKILLS])
        fig = px.bar(sk, x="Count", y="Skill", orientation="h",
                     title=f"Top 20 Skills — {label}",
                     color="Count", color_continuous_scale=color)
        fig.update_layout(yaxis={"categoryorder":"total ascending"}, height=550)
        fig.show()

---
## 6. Track B — Red/Blue State Influence on Data/Tech Employment
**Research Question:** Do state political environments shape salary levels and occupation mix in the data/tech sector?

In [ ]:
# 6.1  Red vs Blue State classification (2024 election)
RED_STATES  = {"AL","AK","AZ","AR","FL","GA","ID","IN","IA","KS","KY",
               "LA","ME","MI","MO","MT","NE","NV","NC","ND","OH","OK",
               "PA","SC","SD","TN","TX","UT","WV","WI","WY"}
BLUE_STATES = {"CA","CO","CT","DE","HI","IL","MD","MA","MN",
               "NH","NJ","NM","NY","OR","RI","VT","VA","WA","DC"}

if "STATE_ABBR" in df.columns:
    df["POLITICAL"] = df["STATE_ABBR"].apply(
        lambda a: "Red State" if a in RED_STATES
                  else ("Blue State" if a in BLUE_STATES else "Swing/Other"))
    print(df["POLITICAL"].value_counts())
else:
    print("STATE_ABBR not available. Run cell 4.8 first.")

In [ ]:
# 6.2  Job postings by political lean
if "POLITICAL" in df.columns:
    pol_counts = df["POLITICAL"].value_counts().reset_index()
    pol_counts.columns = ["Political Lean", "Postings"]
    fig = px.bar(pol_counts, x="Political Lean", y="Postings", color="Political Lean",
                 title="Data/Tech Job Postings by State Political Affiliation (2024)",
                 color_discrete_map={"Red State":"#C62828","Blue State":"#1565C0","Swing/Other":"#6A1B9A"})
    fig.show()

In [ ]:
# 6.3  Salary by political lean
if "POLITICAL" in df.columns and COL_SALARY:
    df_p = df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000)]
    fig = px.box(df_p, x="POLITICAL", y=COL_SALARY, color="POLITICAL",
                 title="Data/Tech Salary: Red vs. Blue vs. Swing States",
                 labels={"POLITICAL": "State Lean", COL_SALARY: "Annual Salary ($)"},
                 color_discrete_map={"Red State":"#C62828","Blue State":"#1565C0","Swing/Other":"#6A1B9A"})
    fig.update_layout(showlegend=False)
    fig.show()
    print(df_p.groupby("POLITICAL")[COL_SALARY]
          .agg(Median="median", Mean="mean", Count="count")
          .sort_values("Median", ascending=False))

In [ ]:
# 6.4  Occupation mix by political lean
if "POLITICAL" in df.columns and COL_OCCUPATION:
    occ_pol = df.groupby(["POLITICAL", COL_OCCUPATION]).size().reset_index(name="Count")
    occ_pol["Share"] = occ_pol.groupby("POLITICAL")["Count"].transform(lambda x: x/x.sum()*100)
    fig = px.bar(occ_pol, x="POLITICAL", y="Share", color=COL_OCCUPATION,
                 barmode="stack",
                 title="Occupation Mix by State Political Lean",
                 labels={"Share": "Share (%)", "POLITICAL": "State Lean"})
    fig.show()

In [ ]:
# 6.5  Remote work by political lean
if "POLITICAL" in df.columns and COL_REMOTE:
    remote_pol = df.groupby(["POLITICAL", COL_REMOTE]).size().reset_index(name="Count")
    remote_pol["Share"] = remote_pol.groupby("POLITICAL")["Count"].transform(lambda x: x/x.sum()*100)
    fig = px.bar(remote_pol, x="POLITICAL", y="Share", color=COL_REMOTE,
                 barmode="stack",
                 title="Remote Work Arrangement by State Political Lean",
                 labels={"Share": "Share (%)", "POLITICAL": "State Lean"})
    fig.show()

In [ ]:
# 6.6  Salary: AI/ML vs. Traditional by political lean
if "POLITICAL" in df.columns and "AI_LABEL" in df.columns and COL_SALARY:
    df_cross = df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000)]
    df_cross = df_cross[df_cross["POLITICAL"] != "Swing/Other"]
    sal_cross = df_cross.groupby(["POLITICAL", "AI_LABEL"])[COL_SALARY].median().reset_index()
    fig = px.bar(sal_cross, x="POLITICAL", y=COL_SALARY, color="AI_LABEL",
                 barmode="group",
                 title="Median Salary: Political Lean x Role Type (AI vs Traditional)",
                 labels={"POLITICAL": "State Lean", COL_SALARY: "Median Salary ($)"},
                 color_discrete_map={"AI / ML": "#1565C0", "Traditional Analyst / BI": "#78909C"})
    fig.show()

---
## 7. Track C — Salary, Geography & Remote Work
**Research Question:** What location and work-arrangement factors best predict salary for data/tech workers?

In [ ]:
# 7.1  Top-paying states
if COL_STATE and COL_SALARY:
    state_sal = (df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000)]
                 .groupby(COL_STATE)[COL_SALARY]
                 .agg(Median="median", Count="count")
                 .reset_index().query("Count >= 30")
                 .sort_values("Median", ascending=False).head(20))
    fig = px.bar(state_sal, x=COL_STATE, y="Median",
                 title="Top 20 States by Median Data/Tech Salary",
                 labels={COL_STATE: "State", "Median": "Median Annual Salary ($)"},
                 color="Median", color_continuous_scale="Viridis")
    fig.update_xaxes(tickangle=-30)
    fig.show()

In [ ]:
# 7.2  Salary choropleth
if "STATE_ABBR" in df.columns and COL_SALARY:
    state_map = (df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000)]
                 .groupby("STATE_ABBR")[COL_SALARY].median().reset_index())
    state_map.columns = ["State", "Median_Salary"]
    fig = px.choropleth(state_map, locations="State", locationmode="USA-states",
                        color="Median_Salary", scope="usa",
                        title="Median Data/Tech Salary by State (2024)",
                        color_continuous_scale="RdYlGn",
                        labels={"Median_Salary": "Median Salary ($)"})
    fig.show()

In [ ]:
# 7.3  Remote salary premium
if COL_REMOTE and COL_SALARY:
    remote_sal = (df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000)]
                  .groupby(COL_REMOTE)[COL_SALARY]
                  .agg(Median="median", Mean="mean", Count="count")
                  .reset_index().sort_values("Median", ascending=False))
    print(remote_sal.to_string(index=False))
    fig = px.bar(remote_sal, x=COL_REMOTE, y="Median",
                 title="Median Salary by Work Arrangement",
                 labels={COL_REMOTE: "Work Type", "Median": "Median Annual Salary ($)"},
                 color="Median", color_continuous_scale="Teal", text="Median")
    fig.update_traces(texttemplate="$%{text:,.0f}", textposition="outside")
    fig.show()

In [ ]:
# 7.4  Remote work trend over time
if COL_REMOTE and "POSTED_MONTH" in df.columns:
    remote_trend = df.groupby(["POSTED_MONTH", COL_REMOTE]).size().reset_index(name="Count")
    remote_trend = remote_trend[remote_trend["POSTED_MONTH"] != "NaT"]
    fig = px.line(remote_trend, x="POSTED_MONTH", y="Count", color=COL_REMOTE,
                  title="Remote / Hybrid / On-Site Posting Trends (2024)", markers=True)
    fig.update_xaxes(tickangle=-45)
    fig.show()

In [ ]:
# 7.5  Experience required vs salary
if COL_EXP and COL_SALARY:
    df_exp = df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000) & df[COL_EXP].notna()]
    fig = px.scatter(df_exp.sample(min(5000, len(df_exp))),
                     x=COL_EXP, y=COL_SALARY,
                     color="AI_LABEL" if "AI_LABEL" in df_exp.columns else None,
                     title="Experience Required vs. Annual Salary (sample 5k)",
                     labels={COL_EXP: "Min. Years Experience", COL_SALARY: "Annual Salary ($)"},
                     opacity=0.4, trendline="ols")
    fig.show()

    df_exp["EXP_BUCKET"] = pd.cut(df_exp[COL_EXP],
                                    bins=[0,2,4,7,10,15,100],
                                    labels=["0-2yr","2-4yr","4-7yr","7-10yr","10-15yr","15+yr"])
    exp_sal = df_exp.groupby("EXP_BUCKET")[COL_SALARY].median().reset_index()
    fig2 = px.bar(exp_sal, x="EXP_BUCKET", y=COL_SALARY,
                  title="Median Salary by Years of Experience Required",
                  labels={"EXP_BUCKET": "Experience Required", COL_SALARY: "Median Salary ($)"},
                  color=COL_SALARY, color_continuous_scale="Viridis")
    fig2.show()

In [ ]:
# 7.6  Top skills overall
if COL_SKILLS:
    from collections import Counter
    all_skills = []
    for v in df[COL_SKILLS].dropna():
        s = str(v)
        sep = next((c for c in [";",",","|"] if c in s), None)
        all_skills.extend([x.strip() for x in s.split(sep)] if sep else [s.strip()])
    top_skills = pd.DataFrame(Counter(all_skills).most_common(30), columns=["Skill","Count"])
    fig = px.bar(top_skills, x="Count", y="Skill", orientation="h",
                 title="Top 30 In-Demand Skills — Data/Tech Jobs (2024)",
                 color="Count", color_continuous_scale="Purples")
    fig.update_layout(yaxis={"categoryorder":"total ascending"}, height=800)
    fig.show()

In [ ]:
# 7.7  Numeric correlation heatmap
num_cols = [c for c in [COL_SALARY, COL_EXP, "DURATION", "MODELED_DURATION",
                         "REMOTE_TYPE", "MIN_EDULEVELS"] if c and c in df.columns]
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    fig = px.imshow(corr, title="Correlation Heatmap — Numeric Features",
                    color_continuous_scale="RdBu", zmin=-1, zmax=1, text_auto=".2f")
    fig.update_layout(height=500)
    fig.show()

---
## 8. Cross-Track: Highest-Paying Job Titles

In [ ]:
if COL_TITLE and COL_SALARY:
    top_pay = (df[(df[COL_SALARY] > 10_000) & (df[COL_SALARY] < 500_000)]
               .groupby([COL_TITLE, "AI_LABEL"])[COL_SALARY]
               .agg(Median="median", Count="count")
               .reset_index().query("Count >= 10")
               .sort_values("Median", ascending=False).head(25))
    fig = px.bar(top_pay, x="Median", y=COL_TITLE, orientation="h", color="AI_LABEL",
                 title="Top 25 Highest-Paying Job Titles (min 10 postings)",
                 labels={"Median": "Median Salary ($)", COL_TITLE: "Job Title"},
                 color_discrete_map={"AI / ML": "#1565C0", "Traditional Analyst / BI": "#78909C"})
    fig.update_layout(yaxis={"categoryorder":"total ascending"}, height=700)
    fig.show()

---
## 9. EDA Summary & Research Question Decision Guide

In [ ]:
print("=" * 68)
print("  AD688 EDA SUMMARY — Key Numbers for Research Question Decision")
print("=" * 68)

print(f"\n[Dataset]")
print(f"  Total postings (cleaned):  {len(df):,}")
print(f"  Scope:                     Data / Analytics / Tech occupations")

if COL_SALARY:
    sal_c = df[(df[COL_SALARY]>10_000)&(df[COL_SALARY]<500_000)][COL_SALARY]
    print(f"  Postings with salary:      {len(sal_c):,} ({len(sal_c)/len(df)*100:.1f}%)")
    print(f"  Median salary:             ${sal_c.median():,.0f}")

if "AI_LABEL" in df.columns:
    ai_n  = (df["AI_LABEL"]=="AI / ML").sum()
    non_n = (df["AI_LABEL"]=="Traditional Analyst / BI").sum()
    print(f"\n[Track A — AI vs. Traditional]")
    print(f"  AI/ML postings:        {ai_n:,} ({ai_n/len(df)*100:.1f}%)")
    print(f"  Traditional postings:  {non_n:,} ({non_n/len(df)*100:.1f}%)")
    if COL_SALARY:
        try:
            df_ai = df[(df[COL_SALARY]>10_000)&(df[COL_SALARY]<500_000)]
            ai_m  = df_ai[df_ai["AI_LABEL"]=="AI / ML"][COL_SALARY].median()
            non_m = df_ai[df_ai["AI_LABEL"]=="Traditional Analyst / BI"][COL_SALARY].median()
            print(f"  AI/ML median salary:   ${ai_m:,.0f}")
            print(f"  Traditional median:    ${non_m:,.0f}")
            print(f"  Salary premium:        ${ai_m-non_m:,.0f} ({(ai_m/non_m-1)*100:+.1f}%)")
        except Exception:
            pass

if "POLITICAL" in df.columns:
    print(f"\n[Track B — Red vs. Blue State]")
    for lean in ["Red State","Blue State","Swing/Other"]:
        n = (df["POLITICAL"]==lean).sum()
        print(f"  {lean:15s}: {n:,} postings ({n/len(df)*100:.1f}%)")
    if COL_SALARY:
        df_p = df[(df[COL_SALARY]>10_000)&(df[COL_SALARY]<500_000)]
        for lean in ["Red State","Blue State"]:
            m = df_p[df_p["POLITICAL"]==lean][COL_SALARY].median()
            print(f"  {lean} median: ${m:,.0f}")

if COL_REMOTE:
    print(f"\n[Track C — Remote Work]")
    for k,v in df[COL_REMOTE].value_counts().items():
        print(f"  {str(k):25s}: {v:,} ({v/len(df)*100:.1f}%)")

print("\n" + "=" * 68)
print("""
RECOMMENDATION GUIDE

Track A — AI vs. Traditional Analyst/BI
  Best if: salary premium is clear AND trend shows AI growth
  Modeling: Random Forest Classification (predict AI vs Non-AI),
             MLR (salary ~ skills + occupation + remote + state)

Track B — Red/Blue State + Tech Employment
  Best if: Blue-Red salary gap >$10k AND occupation mix differs
  Modeling: Logistic Regression (predict state lean from job features),
             MLR (salary ~ political lean + occupation + experience)
  Note: gender proxy is weak in a tech-only dataset.

Track C — Geography / Remote Work
  Best if: strong remote premium OR large state salary variation
  Modeling: MLR (salary ~ state + remote + edu + experience),
             Random Forest Regression
  Simplest modeling path; most data available for regression.

TEAM DECISION: _______________________
""")

---
## 10. Save Cleaned Dataset

In [ ]:
CLEAN_PATH = "lightcast_2024_cleaned.csv"
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved: {CLEAN_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nKey column coverage:")
for name, col in [("Job Title (TITLE_RAW)",COL_TITLE),("Occupation",COL_OCCUPATION),
                   ("Industry (6-digit)",COL_INDUSTRY),("State",COL_STATE),
                   ("Remote Type",COL_REMOTE),("Salary (SALARY_MID)",COL_SALARY),
                   ("Min Education",COL_EDU),("Skills",COL_SKILLS)]:
    if col and col in df.columns:
        n = df[col].notna().sum()
        print(f"  {name:25s} ({col}): {n:,} non-null ({n/len(df)*100:.1f}%)")